# Step 2: Cohort Selection, Outcome Labelling and Feature Engineering (revision 1)

The logic lives in `src/fnpred/cohort.py`. Running `CohortConfig(th_fever=38.5, cancer_flag_mode="ever")`
reproduces the initial-submission dataset exactly (checked on synthetic data).

## Changes from the initial submission
| Item | Initial submission | Revision 1 | Comments |
|---|---|---|---|
| Outcome components | Only `Event` saved | `Event_grade4`, `Event_fn`, `Event_gcsf`, `Event_no_gcsf` (+ `Event_fn_385` for reference) | R1-M1, R2-M2 |
| FN fever threshold | 38.5 °C in code (37.5 °C in the manuscript) | `th_fever = 37.5` (manuscript / JSCO); 38.5 °C kept as `Event_fn_385` | R2-m5 |
| FN timing | Same calendar day (undocumented) | Same calendar day: daily min ANC and daily max temperature (documented) | R2-m5 |
| Exclusions | Silent inner joins | Sequential exclusion with reasons; attrition table; excluded vs. included comparison | R1-M4, R2-M3 |
| Missing values | Described as "no missing values" | Partial missingness (imputed in step 3) reported separately | R1-m6 |
| Cancer-type flags | Whole patient history (incl. future diagnoses) | Diagnoses registered on or before each administration date | R2-m3, R2-m4 |
| Haematological dx after solid tumour | Not examined | Counted; days after the diagnosis can be excluded (`heme_post_dx_policy`) | R2-m4 |

## Unchanged, pending a decision by the authors
- The event window starts on **Day 1**, so a same-day ANC can be both a feature and an outcome (`event_window_start_day`).
- G-CSF is classified as therapeutic / prophylactic **by drug code**.
- The prophylactic G-CSF feature sums doses given on the **2 days** before Day 1 (`pre_gcsf_days`; the manuscript says 3 days).

## Inputs and outputs
- **Input**: `../../data/step1_dwh_outputs/step1_all_*.parquet`
- **Output**: `../../data/step2_processing_outputs/`
  - `step2_all_tagt_feat`
  - `attrition.csv`
  - `all_chemo_days.parquet`
  - `missingness_included.csv`
  - `heme_after_solid_*`
  - `excluded_vs_included.csv`
  - `outcome_components.csv`


In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath("../src"))
import pandas as pd
from fnpred.cohort import CohortConfig, load_step1, build_cohort, save_cohort, compare_excluded_included
pd.set_option("display.max_columns", 50)

In [ ]:
DATA_DIR   = "../../data"
INPUT_DIR  = f"{DATA_DIR}/step1_dwh_outputs"
OUTPUT_DIR = f"{DATA_DIR}/step2_processing_outputs"

cfg = CohortConfig(
    th_fever=37.5,                  # 38.5 reproduces the initial code
    cancer_flag_mode="as_of_date",  # "ever" reproduces the initial code
    heme_post_dx_policy="include",  # "exclude_days": drop administration days on/after a later haematological diagnosis
    event_window_start_day=1,       # pending decision (2 = exclude Day 1 from the outcome window)
)
print(cfg.to_json())

In [ ]:
dfs = load_step1(INPUT_DIR)
res = build_cohort(dfs, cfg)
save_cohort(res, cfg, OUTPUT_DIR)

## Attrition (Fig. 1) and partial missingness

In [ ]:
display(res.attrition)
display(res.missingness)
print(res.all_days["exclusion_reason"].value_counts(dropna=False))

## Excluded vs. included patients (R1-M4, R2-M3)

In [ ]:
cmp = compare_excluded_included(res.all_days)
display(cmp); cmp.to_csv(f"{OUTPUT_DIR}/excluded_vs_included.csv", index=False)

## Outcome components (R2-M2)

In [ ]:
d = res.data
comp = ["Event", "Event_grade4", "Event_fn", "Event_fn_385", "Event_gcsf", "Event_no_gcsf"]
tab = pd.DataFrame({"administration_days": d[comp].sum(), "proportion": d[comp].mean(),
                    "patients_with_>=1": d.groupby("hs")[comp].max().sum()})
# G-CSF-only events (no Grade 4 / FN recorded in the window)
gonly = (d["Event_gcsf"] == 1) & (d["Event_no_gcsf"] == 0)
tab.loc["G-CSF only"] = [gonly.sum(), gonly.mean(), d.loc[gonly, "hs"].nunique()]
display(tab); tab.to_csv(f"{OUTPUT_DIR}/outcome_components.csv")
print("administration days without any ANC/WBC in the 28-day window:", (d["Y_fu_lab_days"] == 0).mean().round(3))

## Haematological diagnoses after a solid tumour (R2-m4)

In [ ]:
print(json.dumps({k: v for k, v in res.heme.items() if k != "top_drugs_after_heme_dx"}, indent=1, ensure_ascii=False))
display(res.heme["top_drugs_after_heme_dx"])

## Reference: initial-submission settings (for the response letter)

In [ ]:
legacy = build_cohort(dfs, CohortConfig(th_fever=38.5, cancer_flag_mode="ever"), verbose=False)
print("initial-submission settings: patients", legacy.data["hs"].nunique(), "days", len(legacy.data),
      "event", round(legacy.data["Event"].mean(), 4))
print("revision settings:           patients", res.data["hs"].nunique(), "days", len(res.data),
      "event", round(res.data["Event"].mean(), 4))